## 案例演示：酒店评论知识库构建

以下代码演示酒店评论场景 3 个向量数据库（评论库、反向 Query 库、摘要库）及 BM25 倒排索引构建

### 环境初始化

In [1]:
import os
import re
import json
import time
import math
import nltk
import jieba
import pickle
import pandas as pd
from pathlib import Path
from collections import Counter
from tqdm.notebook import tqdm

# 下载停用词
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')

# API 客户端
from dashscope import TextEmbedding
import dashvector
from dashvector import Doc
import chromadb

d:\anaconda3\Lib\site-packages\jieba\_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
# 检查环境变量
required_env = {
    "DASHSCOPE_API_KEY": os.getenv("DASHSCOPE_API_KEY"),
    "DASHVECTOR_API_KEY": os.getenv("DASHVECTOR_API_KEY"),
    "DASHVECTOR_HOTEL_ENDPOINT": os.getenv("DASHVECTOR_HOTEL_ENDPOINT"),
}

missing = [k for k, v in required_env.items() if not v]
if missing:
    raise EnvironmentError(f"缺少环境变量: {', '.join(missing)}")

print("环境变量检测成功:")
for key, value in required_env.items():
    print(f"- {key}: {value[:10]}...{value[-4:]}")

环境变量检测成功:
- DASHSCOPE_API_KEY: sk-2824fcf...ca0c
- DASHVECTOR_API_KEY: sk-BSq8DoP...6A67
- DASHVECTOR_HOTEL_ENDPOINT: vrs-cn-il2....com


In [3]:
# Embedding 客户端
class EmbeddingClient:
    """文本嵌入客户端封装"""
    def __init__(self, api_key: str, model: str = "text-embedding-v4", dimension: int = 1024):
        self.api_key = api_key
        self.model = model
        self.dimension = dimension
    
    def embed_batch(self, texts: list[str]) -> list[list[float]]:
        """批量生成 embedding"""
        response = TextEmbedding.call(
            api_key=self.api_key,
            model=self.model,
            input=texts,
            dimension=self.dimension
        )
        
        if response.status_code == 200:
            return [item['embedding'] for item in response.output['embeddings']]
        else:
            raise RuntimeError(f"Embedding 调用失败: {response.message}")

In [4]:
# 初始化 Embedding 客户端
embedding_client = EmbeddingClient(
    api_key=required_env["DASHSCOPE_API_KEY"],
    model="text-embedding-v4",
    dimension=1024
)
BATCH_SIZE = 10  # text-embedding-v4 模型支持的最大 batch size

# 初始化向量数据库客户端
dashvector_client = dashvector.Client(
    api_key=required_env["DASHVECTOR_API_KEY"],
    endpoint=required_env["DASHVECTOR_HOTEL_ENDPOINT"]
)

chroma_db_path = "data/chroma_db"
Path(chroma_db_path).mkdir(parents=True, exist_ok=True)
chroma_client = chromadb.PersistentClient(path=chroma_db_path)
print(f"ChromaDB Path: {chroma_db_path}")

ChromaDB Path: data/chroma_db


In [5]:
# 加载数据
filtered_file = "data/filtered_comments.csv"
queries_file = "data/reverse_queries.csv"
summaries_file = "data/category_summaries.json"

df_filtered = pd.read_csv(filtered_file, index_col=0)
df_queries = pd.read_csv(queries_file)
with open(summaries_file, 'r', encoding='utf-8') as f:
    summaries = json.load(f)

### 1. 构建评论向量库（DashVector）

In [6]:
# DashVector Collection 创建/获取函数
def create_or_get_dashvector_collection(client, name: str, fields_schema: dict = None) -> any:
    """创建或获取 DashVector Collection"""
    collection = client.get(name)
    
    if collection is not None:
        stats = collection.stats()
        code = stats.code if hasattr(stats, "code") else -1
        
        if code == 0:
            print(f"Collection '{name}' 已存在")
            return collection
    
    # 创建新 Collection
    ret = client.create(
        name=name,
        dimension=1024,
        metric="cosine",
        dtype=float,
        fields_schema=fields_schema
    )
    
    if ret and hasattr(ret, "code") and ret.code == 0:
        print(f"Collection '{name}' 创建成功")
    else:
        raise RuntimeError(f"创建 Collection 失败: {ret}")
    
    return client.get(name)

In [7]:
# 创建评论 Collection
comment_fields_schema = {
    'comment': str,
    'room_type': str,
    'fuzzy_room_type': str
}
comment_collection = create_or_get_dashvector_collection(dashvector_client, "comment_database", comment_fields_schema)

Collection 'comment_database' 创建成功


In [8]:
# 生成评论 embedding 并插入
comment_texts = df_filtered['comment'].tolist()
total_inserted = 0

for i in tqdm(range(0, len(comment_texts), BATCH_SIZE), desc="构建评论数据库"):
    batch_texts = comment_texts[i: i + BATCH_SIZE]
    batch_embeddings = embedding_client.embed_batch(batch_texts)
    
    # 构建文档
    docs = []
    for j, embedding in enumerate(batch_embeddings):
        idx = i + j
        row = df_filtered.iloc[idx]
        
        docs.append(Doc(
            id=row.name,
            vector=embedding,
            fields={
                'comment': row['comment'],
                'room_type': row['room_type'],
                'fuzzy_room_type': row['fuzzy_room_type']
            }
        ))
    
    # 插入
    response = comment_collection.upsert(docs)
    if hasattr(response, "code") and response.code == 0:
        total_inserted += len(docs)
    
    time.sleep(0.5)

print(f"评论数据库构建完成, 共更新 {total_inserted} 条记录")

构建评论数据库:   0%|          | 0/218 [00:00<?, ?it/s]

评论数据库构建完成, 共更新 2171 条记录


### 2. 构建反向 Query 向量库（DashVector）

In [9]:
# 创建反向 Query Collection
query_fields_schema = {
    'query': str,
    'comment_id': str,
    'comment': str,
    'room_type': str,
    'fuzzy_room_type': str
}
query_collection = create_or_get_dashvector_collection(dashvector_client, "reverse_query_database", query_fields_schema)

Collection 'reverse_query_database' 创建成功


In [10]:
# 生成 Query embedding 并插入
query_texts = df_queries['query'].tolist()
total_inserted = 0

for i in tqdm(range(0, len(query_texts), BATCH_SIZE), desc="构建反向Query数据库"):
    batch_texts = query_texts[i: i + BATCH_SIZE]
    batch_embeddings = embedding_client.embed_batch(batch_texts)
    
    docs = []
    for j, embedding in enumerate(batch_embeddings):
        idx = i + j
        row = df_queries.iloc[idx]
        
        docs.append(Doc(
            id=f'query_{idx}',
            vector=embedding,
            fields={
                'query': row['query'],
                'comment_id': row['comment_id'],
                'comment': row['comment'],
                'room_type': row['room_type'],
                'fuzzy_room_type': row['fuzzy_room_type']
            }
        ))

    # 插入
    response = query_collection.upsert(docs)
    if hasattr(response, "code") and response.code == 0:
        total_inserted += len(docs)
    
    time.sleep(0.5)

print(f"反向 Query 数据库构建完成, 共更新 {total_inserted} 条记录")

构建反向Query数据库:   0%|          | 0/645 [00:00<?, ?it/s]

反向 Query 数据库构建完成, 共更新 6441 条记录


### 3. 构建摘要向量库（ChromaDB）

In [11]:
# 删除旧数据库
try:
    chroma_client.delete_collection("summary_database")
    print("删除旧的摘要数据库")
except:
    pass

# 创建新数据库
summary_collection = chroma_client.create_collection(
    name="summary_database",
    metadata={'hnsw:space': "cosine"}
)

print("摘要数据库创建成功")

删除旧的摘要数据库
摘要数据库创建成功


In [12]:
# 生成摘要 Embedding 并插入
# 优化：使用「类别名 + 关键词 + 摘要正文前300字」拼接作为 embedding 文本
# 相比只用 keywords（6个短词），向量可充分捕获摘要的语义内容，提升检索相关性
def build_summary_embed_text(s: dict) -> str:
    return f"{s['category']}：{s['keywords']}。{s['summary'][:300]}"

embed_texts = [build_summary_embed_text(s) for s in summaries]

for i in tqdm(range(0, len(embed_texts), BATCH_SIZE), desc="构建摘要数据库"):
    batch_texts = embed_texts[i: i + BATCH_SIZE]
    batch_embeddings = embedding_client.embed_batch(batch_texts)

    num = len(batch_embeddings)
    summary_collection.add(
        ids=[f"summary_{j}" for j in range(i, i + num)],
        embeddings=batch_embeddings,
        documents=[s['summary'] for s in summaries[i: i + num]],
        metadatas=[
            {
                'category': s['category'],
                'keywords': s['keywords'],
                'comment_count': s['comment_count']
            }
            for s in summaries[i: i + num]
        ]
    )

print(f"摘要数据库构建完成, 共更新 {len(summaries)} 条记录")
print("优化说明: embedding 使用「类别名 + 关键词 + 摘要正文前300字」，向量语义更丰富")


构建摘要数据库:   0%|          | 0/2 [00:00<?, ?it/s]

摘要数据库构建完成, 共更新 14 条记录
优化说明: embedding 使用「类别名 + 关键词 + 摘要正文前300字」，向量语义更丰富


In [ ]:
# === 优化：为每个类别生成检索导向的 QA 变体，加入 summary_database ===
#
# 原理：原始摘要是「描述式」长文本，向量与用户问句（「床舒不舒服」「游泳池干净吗」）
#       之间存在 query-document gap。
#       QA 变体是「回答式」短句（20-50 字），embedding 更接近用户问句，
#       命中后 document 字段存完整摘要，LLM 依然获得全量信息。
#
# 运行前提：摘要数据库已在上一个 cell 构建完成（summary_collection 存在）
# 可重复执行：若 QA 变体已存在则跳过，不会重复写入

import time
from dashscope import Generation

def _generate_qa_variants(category: str, summary_text: str) -> list:
    """调用 LLM 为单个类别生成 3 条 QA 检索变体句"""
    prompt = f"""请为酒店点评类别「{category}」生成3个简短的回答式短句，用于检索。要求：
1. 每句 20-50 字，描述该类别的核心事实
2. 风格类似对用户问句的直接回答（如"该酒店游泳池水温适宜，定期消毒，整洁度较好"）
3. 三句分别覆盖：主要优点、需注意的问题、一个具体特色细节

参考摘要（前 500 字）：
{summary_text[:500]}

直接输出 3 句，每行一句，不加序号或符号前缀："""

    try:
        resp = Generation.call(
            model='qwen-turbo',
            messages=[{'role': 'user', 'content': prompt}],
            max_tokens=300,
            temperature=0.4
        )
        if resp.status_code == 200:
            lines = [l.strip() for l in resp.output.text.strip().split('\n') if l.strip()]
            return lines[:3]
    except Exception as e:
        print(f"  [{category}] 生成失败: {e}")
    return []


# 检查是否已有 QA 变体，避免重复写入
try:
    existing_qa = summary_collection.get(where={"type": {"$eq": "qa_variant"}})
    already_exists = len(existing_qa.get('ids', [])) > 0
except Exception:
    already_exists = False

if already_exists:
    print(f"QA 变体已存在 ({len(existing_qa['ids'])} 条)，跳过生成。")
    print("如需重建，请手动删除相关文档后重新运行。")
else:
    qa_id = 0
    failed = []

    for s in tqdm(summaries, desc="生成 QA 变体"):
        variants = _generate_qa_variants(s['category'], s['summary'])

        if not variants:
            failed.append(s['category'])
            continue

        # embed 使用 QA 句子（检索侧），document 存完整摘要（供 LLM 使用）
        variant_embeddings = embedding_client.embed_batch(variants)

        summary_collection.add(
            ids=[f"qa_variant_{qa_id + k}" for k in range(len(variants))],
            embeddings=variant_embeddings,
            documents=[s['summary']] * len(variants),   # 完整摘要供 LLM
            metadatas=[{
                'category':      s['category'],
                'keywords':      s['keywords'],
                'comment_count': s['comment_count'],
                'type':          'qa_variant',
                'qa_text':       variants[k]            # QA 句子记录在 metadata
            } for k in range(len(variants))]
        )
        qa_id += len(variants)
        time.sleep(0.3)  # 避免 API 限流

    print(f"\n✅ QA 变体写入完成：{qa_id} 条")
    if failed:
        print(f"⚠ 以下类别生成失败（可手动重试）: {failed}")
    print(f"summary_database 总文档数: {summary_collection.count()}")
    print(f"  - 原始摘要: 14 条 (summary_0 ~ summary_13)")
    print(f"  - QA 变体:  {qa_id} 条 (qa_variant_0 ~ qa_variant_{qa_id-1})")


### 4. 向量数据库测试

In [13]:
# 验证评论数据库
test_query = "酒店博物馆如何？"
test_embedding = embedding_client.embed_batch([test_query])[0]

results = comment_collection.query(vector=test_embedding, topk=3)

print("评论数据库查询测试:")
print(f"- 查询: {test_query}")
print(f"- 返回结果数: {len(results)}")

print(f"\nTop 1 结果:")
print(f"- 相似度: {results[0].score:.4f}")
print(f"- 评论: {results[0].fields.get('comment', '')}")
print(f"- 房型: {results[0].fields.get('room_type', '')}")

评论数据库查询测试:
- 查询: 酒店博物馆如何？
- 返回结果数: 3

Top 1 结果:
- 相似度: 0.3393
- 评论: 酒店本身就是一个景点，最惊艳的是四楼的博物馆！
- 房型: 花园双床房


In [14]:
# 验证反向 Query 数据库
results = query_collection.query(vector=test_embedding, topk=3)

print("反向 Query 数据库查询测试:")
print(f"- 查询: {test_query}")
print(f"- 返回结果数: {len(results)}")

print(f"\nTop 1 结果:")
print(f"- 相似度: {results[0].score:.4f}")
print(f"- Query: {results[0].fields.get('query', '')}")
print(f"- 关联评论: {results[0].fields.get('comment', '')}")
print(f"- 模糊房型: {results[0].fields.get('fuzzy_room_type', '')}")

反向 Query 数据库查询测试:
- 查询: 酒店博物馆如何？
- 返回结果数: 3

Top 1 结果:
- 相似度: 0.1171
- Query: 酒店博物馆值得去看吗？对住客有没有什么特别待遇？
- 关联评论: 老品牌，颇有底蕴的酒店。给人富丽贵气的感觉。喜欢酒店礼宾部Gary小陈的介绍。相比之下，四楼博物馆门口小哥的态度很让人费解，一个酒店博物馆不就是为了推介吸引参观吗，什么要预约扫码，还不对住客有专享，你以为自己专属文旅就可以这种态度对宾客？搞笑了，让人没胃口看了。
- 模糊房型: 双床房


In [15]:
# 验证摘要数据库
test_query = "酒店博物馆如何？"
test_embedding = embedding_client.embed_batch([test_query])[0]

results = summary_collection.query(query_embeddings=[test_embedding], n_results=1)

print("摘要数据库查询测试:")
print(f"- 查询: {test_query}")
print(f"- 返回结果数: {len(results['ids'][0])}")

# 获取 Top 1 结果
metadata = results['metadatas'][0][0]
document = results['documents'][0][0]

print(f"\nTop 1 结果:")
print(f"- 类别: {metadata.get('category', '')}")
print(f"- 关键词: {metadata.get('keywords', '')}")
print(f"- 摘要: {document[:200]}...")


摘要数据库查询测试:
- 查询: 酒店博物馆如何？
- 返回结果数: 1

Top 1 结果:
- 类别: 公共设施
- 关键词: 岭南园林花园,瀑布与锦鲤,酒店博物馆,金碧辉煌大堂,旋转楼梯与壁画,导览讲解服务
- 摘要: 关于酒店的公共设施，评论普遍给予极高评价，认为其是酒店最核心的亮点与竞争力。主要观点集中在以下几个方面：

首先，酒店内独具特色的岭南园林花园是几乎所有评论都提及的焦点。住客们惊叹于在市中心竟能拥有如此精致、静谧且打理用心的花园，其中人工双瀑布、小桥流水、池塘与成群锦鲤构成了核心景观，被誉为“城市绿洲”和“世外桃源”，特别适合亲子游玩、散步与拍照，完美诠释了酒店名称。

其次，酒店内部充满艺术与文...


### 5. 倒排索引构建与测试

In [16]:
# 倒排索引类定义
class InvertedIndex:
    """基于 BM25 的倒排索引"""
    
    def __init__(self, k1: float = 1.5, b: float = 0.75, stopwords_file: str = None):
        """
        参数:
            k1: BM25 参数，控制词频饱和度
            b: BM25 参数，控制文档长度归一化程度
        """
        self.k1 = k1
        self.b = b
        self.index = {}          # {term: {doc_id: term_freq}}
        self.doc_lengths = {}    # {doc_id: doc_length}
        self.avg_doc_length = 0
        self.num_docs = 0
        self.documents = {}      # {doc_id: document_content}

        # 加载停用词
        self.stopwords = set()
        if stopwords_file and Path(stopwords_file).exists():
            with open(stopwords_file, encoding='utf-8') as f:
                self.stopwords.update([line.strip() for line in f])
            try:
                self.stopwords.update(nltk.corpus.stopwords.words('english'))  # 加载英文停用词
            except:
                print("警告: 未能加载 NLTK 英文停用词")
        
        # 字典预加载
        jieba.initialize()
    
    def tokenize(self, text: str) -> list[str]:
        """分词与过滤"""     
        
        # 删除空白字符
        text = re.sub(r'\s+', '', text)
        
        # 中文分词
        tokens = jieba.lcut(text)
        
        # 过滤停用词、非中英文字符，统一小写
        pattern = re.compile(r'[^\u4e00-\u9fffa-zA-Z]')
        tokens = [token.lower() for token in tokens if token.lower() not in self.stopwords and not pattern.search(token)]
        
        return tokens
    
    def build(self, documents: dict[str, str]):
        """
        构建倒排索引
        
        参数:
            documents: {doc_id: document_text}
        """
        self.documents = documents
        self.num_docs = len(documents)
        
        # 统计文档长度
        total_length = 0
        for doc_id, text in tqdm(documents.items(), desc="分词与统计"):
            tokens = self.tokenize(text)
            doc_length = len(tokens)
            self.doc_lengths[doc_id] = doc_length
            total_length += doc_length
            
            # 构建倒排索引
            term_freq = Counter(tokens)
            for term, freq in term_freq.items():
                if term not in self.index:
                    self.index[term] = {}
                self.index[term][doc_id] = freq
        
        self.avg_doc_length = total_length / self.num_docs if self.num_docs > 0 else 0
        print(f"倒排索引构建完成: {len(self.index)} 个词项, {self.num_docs} 篇文档")
        print(f"平均文档长度: {self.avg_doc_length:.2f} 个词")
    
    def search(self, query: str, topk: int = 10) -> list[tuple[str, float]]:
        """
        BM25 检索
        
        参数:
            query: 查询文本
            topk: 返回 Top-K 结果
        
        返回:
            [(doc_id, bm25_score), ...]
        """
        query_tokens = self.tokenize(query)

        if not query_tokens:
            return []
        
        # 计算 IDF
        idf = {}
        for term in query_tokens:
            if term in self.index:
                df = len(self.index[term])  # 文档频率
                idf[term] = math.log((self.num_docs - df + 0.5) / (df + 0.5) + 1.0)
        
        # 计算 BM25 分数
        scores = {}
        for term in query_tokens:
            if term not in self.index:
                continue
            
            for doc_id, tf in self.index[term].items():
                if doc_id not in scores:
                    scores[doc_id] = 0
                
                doc_length = self.doc_lengths[doc_id]
                norm_factor = 1 - self.b + self.b * (doc_length / self.avg_doc_length)
                term_score = idf[term] * (tf * (self.k1 + 1)) / (tf + self.k1 * norm_factor)
                scores[doc_id] = scores.get(doc_id, 0) + term_score
        
        # 排序并返回 Top-K
        sorted_docs = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:topk]
        return sorted_docs
    
    def save(self, filepath: str):
        """保存索引到文件"""
        with open(filepath, 'wb') as f:
            pickle.dump({
                'index': self.index,
                'doc_lengths': self.doc_lengths,
                'avg_doc_length': self.avg_doc_length,
                'num_docs': self.num_docs,
                'documents': self.documents,
                'k1': self.k1,
                'b': self.b,
                'stopwords': self.stopwords
            }, f)
        print(f"倒排索引已保存: {filepath}")
    
    def load(self, filepath: str):
        """从文件加载索引"""
        with open(filepath, 'rb') as f:
            data = pickle.load(f)
            self.index = data['index']
            self.doc_lengths = data['doc_lengths']
            self.avg_doc_length = data['avg_doc_length']
            self.num_docs = data['num_docs']
            self.documents = data['documents']
            self.k1 = data['k1']
            self.b = data['b']
            self.stopwords = data.get('stopwords', set())
        print(f"倒排索引已加载")

In [17]:
# 构建倒排索引
stopwords_file = "data/stopwords_chinese.txt"                                  # 中文停用词文件路径
inverted_index = InvertedIndex(k1=1.5, b=0.75, stopwords_file=stopwords_file)
documents = {idx: row['comment'] for idx, row in df_filtered.iterrows()}       # 使用 comment_id 作为 doc_id
inverted_index.build(documents)

# 保存倒排索引
index_file = "data/inverted_index.pkl"
inverted_index.save(index_file)

Building prefix dict from the default dictionary ...
Loading model from cache C:\Users\26524\AppData\Local\Temp\jieba.cache


Loading model cost 0.584 seconds.
Prefix dict has been built successfully.


分词与统计:   0%|          | 0/2171 [00:00<?, ?it/s]

倒排索引构建完成: 10734 个词项, 2171 篇文档
平均文档长度: 41.62 个词
倒排索引已保存: data/inverted_index.pkl


In [18]:
# 加载倒排索引
index_file = "data/inverted_index.pkl"
inverted_index = InvertedIndex()
inverted_index.load(index_file)

# 验证倒排索引
test_query_bm25 = "酒店博物馆和瀑布花园如何？"
bm25_results = inverted_index.search(test_query_bm25, topk=3)

print("\n倒排索引查询测试:")
print(f"- 原始查询: {test_query_bm25}")
print(f"- 分词结果: {inverted_index.tokenize(test_query_bm25)}")
print(f"- 返回结果数: {len(bm25_results)}")

# 获取 Top 1 结果
doc_id, score = bm25_results[0]
comment = df_filtered.loc[doc_id]

print(f"\nTop 1 结果:")
print(f"- BM25 分数: {score:.4f}")
print(f"- 评论: {comment['comment']}")

倒排索引已加载

倒排索引查询测试:
- 原始查询: 酒店博物馆和瀑布花园如何？
- 分词结果: ['酒店', '博物馆', '瀑布', '花园']
- 返回结果数: 3

Top 1 结果:
- BM25 分数: 7.9787
- 评论: 酒店就是住在花园里，亮点是花园瀑布下午茶，还有4楼博物馆也可免费参观，很不错的体验哦
